In [ ]:
#source ~/venv_gcepy/bin/activate
#환경 활성화 후 실

In [ ]:
import numpy, jax, gcepy
print('numpy', numpy.__version__, '| jax', jax.__version__)
print('gcepy data shapes:')
print('  fermi_front_20x20:', gcepy._lm.fermi_front_20x20.shape)
print('  mask_20x20:       ', gcepy._lm.mask_20x20.shape)
print('  bremss_8t:        ', gcepy._lm.bremss.shape)
print('  prior log10(c) low :', gcepy._lm.pmin)
print('  prior log10(c) high:', gcepy._lm.pmax)

In [ ]:
import sys, time, jax.numpy as jnp, jax.random as jr, numpy as np
import dynesty as dn, numpyro as nr, gcepy as gp

EBIN, EXNUM, PDIM = 7, 1, 5
rng_key = jr.PRNGKey(0)
lo, hi = np.asarray(gp._lm.pmin[:PDIM]), np.asarray(gp._lm.pmax[:PDIM])
ptform = lambda x: x * (hi - lo) + lo

print(f'=== bin {EBIN}, ex_num={EXNUM}, PDIM={PDIM} ===')
print(f'prior low (log10 c):  {lo}')
print(f'prior high (log10 c): {hi}')
print(f'init lnlike @ c=1: {float(gp.lnlike("low", jnp.zeros(PDIM), bin_no=EBIN, ex_num=EXNUM)):.2f}\n')

# dynesty
print('=== dynesty NestedSampler (dlogz=10) ===', flush=True)
t0 = time.time()
sampler_dn = dn.NestedSampler(
    lambda x: float(gp.lnlike('low', x, EBIN, ex_num=EXNUM)),
    ptform, PDIM)
sampler_dn.run_nested(dlogz=10, print_progress=False)
dn_logl = np.asarray(sampler_dn.results.logl)
i_best = int(np.argmax(dn_logl))
theta_dn = np.asarray(sampler_dn.results.samples[i_best])
c_dn = 10**theta_dn
print(f'  done in {time.time()-t0:.1f}s')
print(f'  best logl  = {float(dn_logl[i_best]):+.4f}')
print(f'  best logz  = {float(sampler_dn.results.logz[-1]):+.4f}')
print(f'  best theta = {theta_dn}')
print(f'  best c     = PB={c_dn[0]:.4f}  ICS={c_dn[1]:.4f}  Bub={c_dn[2]:.4f}  Iso={c_dn[3]:.4f}  GCE={c_dn[4]:.4f}')

# NUTS
print('\n=== numpyro NUTS (num_samples=10000, warmup=100) ===', flush=True)
t0 = time.time()
sampler_nr = nr.infer.MCMC(
    nr.infer.NUTS(potential_fn=lambda t: -gp.lnlike('low', t, bin_no=EBIN, ex_num=EXNUM)),
    num_samples=10000, num_warmup=100,
    jit_model_args=True, chain_method='vectorized', progress_bar=False)
sampler_nr.run(rng_key, init_params=jnp.zeros(PDIM))
samples_nr = np.asarray(sampler_nr.get_samples())
lls_nr = np.asarray([float(gp.lnlike('low', s, bin_no=EBIN, ex_num=EXNUM)) for s in samples_nr])
i_best_nr = int(np.argmax(lls_nr))
theta_nr = samples_nr[i_best_nr]
c_nr = 10**theta_nr
print(f'  done in {time.time()-t0:.1f}s')
print(f'  best logl  = {float(lls_nr[i_best_nr]):+.4f}')
print(f'  best theta = {theta_nr}')
print(f'  best c     = PB={c_nr[0]:.4f}  ICS={c_nr[1]:.4f}  Bub={c_nr[2]:.4f}  Iso={c_nr[3]:.4f}  GCE={c_nr[4]:.4f}')

# 비교 (paper code default = Model 8t = XLIX)
print('\n=== 비교 ===')
print(f'                  {"PB":>8} {"ICS":>8} {"Bub":>8} {"Iso":>8} {"GCE":>8}')
print(f'  paper dynesty : {c_dn[0]:>8.3f} {c_dn[1]:>8.3f} {c_dn[2]:>8.3f} {c_dn[3]:>8.3f} {c_dn[4]:>8.3f}  (Model 8t = XLIX, bin 7)')
print(f'  paper NUTS    : {c_nr[0]:>8.3f} {c_nr[1]:>8.3f} {c_nr[2]:>8.3f} {c_nr[3]:>8.3f} {c_nr[4]:>8.3f}')
print(f'  cell 8q digit : {1.001:>8.3f} {0.819:>8.3f} {0.340:>8.3f} {1.610:>8.3f} {3.130:>8.3f}  (cell 8q v3 Model I digitize)')
print(f'  우리 emcee XLIX: results_12yr/GCE_model_XLIX_*.npz 의 bin 7 fitted_params 별도 load 필요')

In [ ]:
import numpy as np

# 우리 emcee Model XLIX bin 7 결과 load + paper code 와 비교
d_xlix = np.load('./results_12yr/GCE_model_XLIX_front_12yr_cholis_fit.npz')
print('=== 우리 emcee Model XLIX (= paper 8t) bin 7 ===\n')
print(f'  fitted (max-lh): PB={d_xlix["fitted_params"][0,7]:.4f}  '
      f'ICS={d_xlix["fitted_params"][1,7]:.4f}  '
      f'GCE={d_xlix["fitted_params"][2,7]:.4f}  '
      f'Bub={d_xlix["fitted_params"][3,7]:.4f}  '
      f'Iso={d_xlix["fitted_params"][4,7]:.4f}')
print(f'  median:          PB={d_xlix["fitted_params_median"][0,7]:.4f}  '
      f'ICS={d_xlix["fitted_params_median"][1,7]:.4f}  ...')
print(f'  std:             PB={d_xlix["fitted_params_std"][0,7]:.4f}  '
      f'ICS={d_xlix["fitted_params_std"][1,7]:.4f}')
print(f'  max_logL[7] = {d_xlix["max_likelihood"][7]:.4f}')
print(f'  → -2 × max_logL = {-2*d_xlix["max_likelihood"][7]:.4f}')

print(f'\n=== paper code (방금 dynesty) Model 8t bin 7 ===')
print(f'  best lnlike = -167637.28  →  -2 × lnL = +335274.56')
print(f'  c = PB=0.692  ICS=0.929  Bub=0.542  Iso=0.0101  GCE=0.870')

print(f'\n=== 비교 ===')
our_2logL  = -2 * d_xlix['max_likelihood'][7]
paper_2logL = -2 * (-167637.28)
print(f'  -2logL diff (ours - paper) = {our_2logL - paper_2logL:+.2f}')
print(f'  → 양수면 우리 emcee 가 worse fit (mode trap), 음수/0 이면 우리 가 같은 또는 better mode')

In [ ]:
# paper code _lm ndarray inventory + dump (fermi kernel 에서 load 용)
import numpy as np
import gcepy as gp

EBIN = 7

print('=== paper code (gcepy._lm) ndarray attributes ===')
to_dump = {}
for attr in sorted(dir(gp._lm)):
    if attr.startswith('_'):
        continue
    obj = getattr(gp._lm, attr)
    if not isinstance(obj, np.ndarray):
        continue
    to_dump[attr] = obj
    if obj.ndim >= 1 and obj.shape[0] == 14:
        print(f'  {attr:30s} shape={str(obj.shape):18s} dtype={str(obj.dtype):8s} '
              f'sum(bin{EBIN})={obj[EBIN].sum():.6e}')
    else:
        print(f'  {attr:30s} shape={str(obj.shape):18s} dtype={str(obj.dtype):8s} '
              f'sum={obj.sum():.6e}')

OUT = '/tmp/gcepy_lm_dump.npz'
np.savez(OUT, **to_dump)
print(f'\n→ dumped to {OUT}  ({len(to_dump)} arrays)')

In [ ]:
# paper code _lm ndarray inventory + dump (fermi kernel 에서 load 용)
# gp._lm.dir() 가 빈 list 인 경우 대비: hardcoded names + filesystem walk
import os
import numpy as np
import gcepy as gp

EBIN = 7

# Method 1: hardcoded attribute names on gp._lm
# cell 1 에서 fermi_front_20x20, mask_20x20, bremss_8t 확인됨
# paper code default = Model 8t = XLIX
known = [
    'fermi_front_20x20', 'fermi_back_20x20',
    'mask_20x20',
    'bremss', 'bremss_8t', 'pion', 'pion_8t', 'gas', 'gas_8t', 'pb',
    'ics', 'ics_8t', 'ic',
    'bubble', 'bubble_8t', 'bub', 'fermi_bubble',
    'iso', 'iso_8t', 'isotropic',
    'gce', 'gce_8t', 'wimp', 'nfw2', 'GCE',
]
print('=== Method 1: gp._lm hardcoded attribute scan ===')
paper = {}
for attr in known:
    try:
        obj = getattr(gp._lm, attr)
    except AttributeError:
        continue
    if not isinstance(obj, np.ndarray):
        continue
    paper[attr] = obj
    if obj.ndim >= 1 and obj.shape[0] == 14:
        print(f'  {attr:25s} shape={str(obj.shape):20s} dtype={str(obj.dtype):8s} '
              f'sum(bin{EBIN})={obj[EBIN].sum():.6e}')
    else:
        print(f'  {attr:25s} shape={str(obj.shape):20s} dtype={str(obj.dtype):8s} '
              f'sum={obj.sum():.6e}')

# Method 2: filesystem walk for .npy in gcepy/inputs/
print('\n=== Method 2: gcepy/inputs/ .npy files ===')
gcepy_dir  = os.path.dirname(gp.__file__)
inputs_dir = os.path.join(gcepy_dir, 'inputs')
print(f'inputs_dir: {inputs_dir}')
if os.path.isdir(inputs_dir):
    for root, dirs, files in os.walk(inputs_dir):
        for f in sorted(files):
            if not f.endswith('.npy'):
                continue
            full = os.path.join(root, f)
            rel  = os.path.relpath(full, inputs_dir)
            try:
                arr = np.load(full, allow_pickle=False)
            except Exception as e:
                print(f'  {rel}: load failed ({type(e).__name__})')
                continue
            key = rel.replace('/', '__').replace('.npy', '')
            if key not in paper:
                paper[key] = arr
            if arr.ndim >= 1 and arr.shape[0] == 14:
                print(f'  {rel:55s} shape={str(arr.shape):20s} sum(bin{EBIN})={arr[EBIN].sum():.6e}')
            else:
                print(f'  {rel:55s} shape={str(arr.shape):20s} sum={arr.sum():.6e}')
else:
    print(f'  (디렉토리 없음)')

OUT = '/tmp/gcepy_lm_dump.npz'
np.savez(OUT, **paper)
print(f'\n→ dumped {len(paper)} arrays to {OUT}')

In [ ]:
# paper code source 직접 확인
# 목적:
#  1) ex_num indexing: excesses/ 의 4 후보 (bb/bbp/dm/x) 중 ex_num=1 이 어느 것인가
#  2) PB 처리 확인: pion0+bremss 가 fit 시 단일 c_PB 인지 (이미 cell 2 PDIM=5 로 추정 확인됨)
#  3) likelihood 식 (chi2 external 포함 여부 / Eq.13 매칭)
import inspect
import gcepy as gp

# (1) gp.lnlike source
print('=' * 70)
print('gp.lnlike source')
print('=' * 70)
try:
    print(inspect.getsource(gp.lnlike))
except (OSError, TypeError) as e:
    print(f'  inspect.getsource failed ({type(e).__name__}: {e})')
    print(f'  type(gp.lnlike) = {type(gp.lnlike)}')
    print(f'  gp.lnlike = {gp.lnlike}')

# (2) gp._lm module source
print('=' * 70)
print(f'gp._lm module file: {inspect.getsourcefile(gp._lm)}')
print('=' * 70)
try:
    print(inspect.getsource(gp._lm))
except (OSError, TypeError) as e:
    print(f'  inspect.getsource failed ({type(e).__name__}: {e})')
    # fallback: file 직접 read
    src_file = inspect.getsourcefile(gp._lm)
    if src_file:
        with open(src_file) as f:
            print(f.read())

# (3) gp 의 top-level 노출 (lnlike 외에 다른 함수 있는지)
print('=' * 70)
print('gp module top-level names')
print('=' * 70)
for name in sorted(dir(gp)):
    if name.startswith('_'):
        continue
    obj = getattr(gp, name)
    print(f'  {name:20s} {type(obj).__name__}')

In [ ]:
# paper code likelihood component breakdown (Poisson + chi2_bub + chi2_iso)
# paper jmodel order: [PB, ICS, Bub, Iso, GCE]
import numpy as np
import jax.numpy as jnp
import jax.scipy.special as jsc
import gcepy as gp

EPS = jnp.finfo(jnp.float32).eps

def paper_components(c_dict, bin_no=7, ex_num=1):
    theta = jnp.log10(jnp.asarray(
        [c_dict['PB'], c_dict['ICS'], c_dict['Bub'], c_dict['Iso'], c_dict['GCE']]
    ))
    e = gp._lm.jjmodel_masked(theta, bin_no, ex_num)
    d = gp._lm.jjdata_masked(bin_no)
    mylike   = 2 * float(jnp.sum(e + jsc.gammaln(d+1) - jsc.xlogy(d, e+EPS)))
    chi2_bub = float(((10**theta[2] - 1.0) / gp._lm.bubble_error[bin_no])**2)
    chi2_iso = float(((10**theta[3] - 1.0) / gp._lm.isotropic_error[bin_no])**2)
    return mylike, chi2_bub, chi2_iso

c_paper = {'PB':0.6918, 'ICS':0.9295, 'Bub':0.5418, 'Iso':0.0101, 'GCE':0.8698}
c_ours  = {'PB':0.4025, 'ICS':1.7423, 'Bub':0.0874, 'Iso':0.0005, 'GCE':2.9230}

print(f'  bin 7, paper input + paper likelihood, ex_num=1 (DM)')
print(f'  external_errors[bin 7]: iso={float(gp._lm.isotropic_error[7]):.4f}  '
      f'bub={float(gp._lm.bubble_error[7]):.4f}')
print()
print(f'{"c choice":<22} {"Poisson":>14} {"chi2_bub":>10} {"chi2_iso":>10} {"total -2logL":>14}')
print('-' * 75)
for label, c in [('paper best c', c_paper), ('우리 best c', c_ours)]:
    p, b, i = paper_components(c, bin_no=7, ex_num=1)
    print(f'{label:<22} {p:>14.2f} {b:>10.4f} {i:>10.4f} {p+b+i:>14.2f}')